# Welcome to the Sandbox.
## A place to write and test random stuff

Double click into cells to edit them, this will transform markdown cells from their nicely formatted versions to the actual markdown script the first thing that we have here is a series of basic python scripts to make sure everything is running as expected.

#### The code below is just simple processing on a given list of integers. It makes use of python's ability to simplify 'and' expressions for integer ranges. Very nice.

In [ ]:
#Gatekeeper Code Exercise
population_ages = [10,4,18,12,16,20,26,100,24]
for age in population_ages:
    if age <= 6:
        print('You are ' + str(age) + ' years old and cannot enter')
    elif 6 < age <= 12:
        print('You are ' + str(age) + ' years old and can enter child friendly areas')
    elif 12 < age <= 16:
        print('You are ' + str(age) + ' years old and can enter with an adult')
    elif age >= 17:
        print('You are ' + str(age) + ' years old and can enter freely')
    else:
        print('somehow palpatine returned?')

## Challenge:
Write a login program that asks a user for username and password, which are predefined. Full brief [**here**](https://vle.aston.ac.uk/ultra/courses/_62426_1/outline/edit/document/_4285221_1?courseId=_62426_1&view=content&state=view)


In [ ]:
#set up a login bools for while loop
good_login:bool = False
attempts:int = 0
while not good_login and attempts < 3:
    #Request username
    username:str = input("Enter your username: ")
    #Request password
    password:str = input("Enter your password: ")
    if username == 'admin' and password == 'password123':
        print('You are logged in')
        good_login = True
    elif username.lower() == 'quit' and password.lower() == '':
        break
    else:
        attempts += 1
        if attempts < 3:
            print("You are not logged in and have %d attempts left, check your username and password and try again or enter username: QUIT password: *blank* to exit the application" % (3-attempts))
print('Allowed attempts exceeded or an error occurred. Please try again later')

# Course Work Data Playground

## Imports

In [ ]:
import pandas as pd
import pandasql as ps
import json
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler

## Playground

In [ ]:
with open('ingest_schema.json') as f:
    schema = json.load(f)

#Read in the whole CSV.
#df_all_str = pd.read_csv("CW_data.csv",dtype = str, encoding='ANSI')
#df_all.shape
#df_all_str[df_all_str.apply(lambda col: col.str.contains('FALSE')).any(axis=1)]

df_all = pd.read_csv(
    filepath_or_buffer ="CW_data.csv",
    encoding='ANSI',
    dtype = schema,
    true_values = ["positive"],
    false_values = ["negative"],
    on_bad_lines = "warn"
    )
print(df_all.dtypes)


## Handling The Non-Numerics

In [ ]:
#deal with non-numeric data points

In [ ]:
df_all.head(10)

Seems to be some useless columns with just values of NA. I could find a way to drop columns with all NA, but it might be worth dropping columns with all values that are identical, regardless of if they are NaN or not. I think this is the correct choice as a column with all the same value will likely not provide any good information for the model to interpret and find trends. It would also lead to issues when looking at the test data set as if a col in the development data is full of a single value but in the test data it has multiple values, the algorithm will not have sufficient knowledge to assess the weight of those other differing values. Therefore, exclude all cols that have only got one unique value.

In [ ]:
missing_per_column = df_all.isnull().sum(axis=0)
plt.plot(missing_per_column)
df_single_values =  df_all.loc[:, df_all.nunique(dropna = False) == 1]
print(df_single_values.columns)

In [ ]:
df_single_values.head()

In [ ]:
#df_single_values.columns
df_filtered = df_all.drop(labels = df_single_values.columns, axis = 1)
pd.set_option('display.max_rows', None)
print(df_filtered.dtypes)

In [ ]:
missing_per_column = df_filtered.isnull().sum(axis=0)
#print(missing_per_column)
plt.plot(missing_per_column)

In [ ]:
missing_per_row = df_filtered.isnull().sum(axis=1)
plt.plot(abs(missing_per_row))
print(missing_per_row)

In [ ]:
df_numerics = df_filtered.select_dtypes(include = 'number')
print(df_numerics.columns)

## Attempting PCA
### this needs non numerical columns to be dealt with first and NANs to be handled

In [ ]:
#Grab classifiers
df_classifiers = df_filtered['SARS-Cov-2 exam result']
print(f'classifiers shape is {df_classifiers.shape}')
#deal with NANs in the numerical - this will need to be done properly with outlier handling, but for now, fuck it. MEAN TIME BABY
df_no_nan_numerics = df_numerics.fillna(df_numerics.mean())

#Scale this shit down
scaler = StandardScaler()
df_scaled_predictors = scaler.fit_transform(df_no_nan_numerics)
print(f'scaled predictor shape is: {df_scaled_predictors.shape}')

#create a pca and fit that bitch up
pca = PCA(n_components = 10)
predictor_pca = pca.fit_transform(df_scaled_predictors)

#split em
predictor_train, predictor_test, classifier_train, classifier_test = train_test_split(predictor_pca, df_classifiers, test_size = 0.2)

#predict em
model = LogisticRegression()
model.fit(predictor_train, classifier_train)
classifier_pred = model.predict(predictor_test)

#visualise em
cm = confusion_matrix(classifier_test, classifier_pred)
plt.figure(figsize = (10,10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Positive', 'Negative'], yticklabels=['Positive', 'Negative'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()